# MLflow — trace, evaluate and package your agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USER/SA_LIB_WORKSHOP_Day2/blob/main/06_mlflow/06_mlflow.ipynb)

**SA-LIB AI Workshop 2026 — Day 2: AI Agents**

MLflow is not an agent framework: it's the **lab notebook** for all of them. Tracing shows every LLM and tool call, evaluation scores your agent on a test set, and packaging turns it into a versioned model you can serve.

In Colab, look at traces with `mlflow.search_traces()` (the UI needs a local machine or a tunnel).

Same example in every framework: **(1) first agent → (2) tools → (3) memory → (4) fallback → (5) multi-agent → complete agent**.

## 0. Setup
Install the packages (takes ~1 minute in Colab).

In [ ]:
%pip install -q "mlflow>=3.0" "langgraph>=1.0" "langchain-openai>=1.0" "openai" "python-dotenv" "ddgs"

### API keys
* **Colab:** click the 🔑 *Secrets* icon on the left, add e.g. `GROQ_API_KEY`, and enable notebook access.
* **VS Code / Jupyter / Linux:** put your keys in the `.env` file in the repo root (copy `.env.example`).
* Otherwise you'll be asked to paste a key below.

Choose the model provider with `LLM_PROVIDER`: `groq` · `gemini` · `openai` · `deepseek` · `openrouter` · `ollama`.

In [ ]:
import os, getpass
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # VS Code / Linux: read ../.env
try:                                           # Colab: read the Secrets panel
    from google.colab import userdata
    for key in ['GROQ_API_KEY', 'GOOGLE_API_KEY', 'OPENAI_API_KEY', 'DEEPSEEK_API_KEY', 'OPENROUTER_API_KEY', 'LANGSMITH_API_KEY', 'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_HOST', 'LLM_PROVIDER', 'LLM_MODEL']:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("LLM_PROVIDER", "groq")   # <- change the provider here if you like
provider = os.environ["LLM_PROVIDER"]
key_name = {"groq": "GROQ_API_KEY", "gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY",
            "deepseek": "DEEPSEEK_API_KEY", "openrouter": "OPENROUTER_API_KEY"}.get(provider)
if key_name and not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"{key_name}: ")
print("provider:", provider)

### Shared helper: `workshop_common.py`
Reads the provider settings, and holds the two tools every example uses: `web_search` (free DuckDuckGo) and `word_count`.

In [ ]:
%%writefile workshop_common.py
"""
workshop_common.py — shared helpers for every framework folder in this workshop.

(the SAME file is copied into each folder so every folder works on its own —
 edit shared/workshop_common.py and run `python tools/sync_common.py` to update the copies)

What it gives you:
  get_llm_config()   -> which provider / model / base_url / api key to use (read from .env)
  fallback_order()   -> the list of providers to try, in order, for the fallback examples
  ping()             -> quick "are you alive?" check against a provider
  web_search()       -> free DuckDuckGo search (no key)
  word_count()       -> the tiny "tool you wrote yourself" from the original guide

Every provider below speaks the OpenAI chat-completions protocol, which is why every
framework in this workshop can use every provider: we just change base_url + api_key + model.
"""

from __future__ import annotations

import os
from dataclasses import dataclass

from dotenv import find_dotenv, load_dotenv

# walks up from where you run python, so the .env in the repo root is always found
load_dotenv(find_dotenv(usecwd=True))

PROVIDERS = {
    #  name        base_url                                                     api-key env var       default model
    "groq":       ("https://api.groq.com/openai/v1",                           "GROQ_API_KEY",       "llama-3.3-70b-versatile"),
    "gemini":     ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY",     "gemini-2.5-flash"),
    "openai":     ("https://api.openai.com/v1",                                "OPENAI_API_KEY",     "gpt-4o-mini"),
    "deepseek":   ("https://api.deepseek.com",                                 "DEEPSEEK_API_KEY",   "deepseek-chat"),
    "openrouter": ("https://openrouter.ai/api/v1",                             "OPENROUTER_API_KEY", "meta-llama/llama-3.3-70b-instruct:free"),
    "ollama":     (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),  None,                 "qwen2.5:7b"),
}


@dataclass
class LLMConfig:
    provider: str
    model: str
    base_url: str
    api_key: str

    @property
    def litellm_model(self) -> str:
        """Model name in LiteLLM format (used by CrewAI and Google ADK): 'openai/<model>'
        means 'an OpenAI-compatible endpoint' — combined with base_url it works for every provider."""
        return f"openai/{self.model}"


def get_llm_config(provider: str | None = None) -> LLMConfig:
    """Read LLM_PROVIDER / LLM_MODEL / <PROVIDER>_API_KEY from the environment."""
    provider = (provider or os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown LLM_PROVIDER '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    base_url, key_env, default_model = PROVIDERS[provider]
    api_key = os.getenv(key_env) if key_env else "ollama"  # ollama runs locally, no key
    if not api_key:
        raise RuntimeError(f"{key_env} is missing — add it to your .env file (see .env.example).")
    # LLM_MODEL only overrides the model of the MAIN provider, not of the fallbacks
    main_provider = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    model = (os.getenv("LLM_MODEL") if provider == main_provider else None) or default_model
    return LLMConfig(provider, model, base_url, api_key)


def fallback_order() -> list[str]:
    """Providers to try, in order. Default: your LLM_PROVIDER first, then LLM_FALLBACKS (groq,gemini)."""
    main = (os.getenv("LLM_PROVIDER") or "groq").strip().lower()
    extra = [p.strip().lower() for p in os.getenv("LLM_FALLBACKS", "groq,gemini").split(",") if p.strip()]
    return list(dict.fromkeys([main, *extra]))  # remove duplicates, keep order


def ping(cfg: LLMConfig) -> None:
    """Send a 1-token request. Raises an exception if the provider is down, rate-limited or the key is wrong."""
    from openai import OpenAI

    OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=20, max_retries=0).chat.completions.create(
        model=cfg.model, messages=[{"role": "user", "content": "ping"}], max_tokens=1
    )


def web_search(query: str) -> str:
    """Search the web with DuckDuckGo (free, no key) and return the top results as text."""
    try:
        from ddgs import DDGS

        results = DDGS().text(query, max_results=3)
        return "\n".join(f"- {r['title']}: {r['body']} ({r['href']})" for r in results) or "no results"
    except Exception as e:  # the free search is rate-limited: return the error instead of crashing
        return f"web search failed ({e}). try again in a few seconds."


def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### helper file: `agent_model.py`
the agent that 04_package_agent.py logs as an MLflow model ("models from code").
MLflow's ResponsesAgent interface = a standard input/output format, so the model can be
served behind a REST API (mlflow models serve) no matter which framework is inside.

In [ ]:
%%writefile agent_model.py
# the agent that 04_package_agent.py logs as an MLflow model ("models from code").
# MLflow's ResponsesAgent interface = a standard input/output format, so the model can be
# served behind a REST API (mlflow models serve) no matter which framework is inside.

import json
import uuid

import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse
from openai import OpenAI

import workshop_common as wc

SPECS = [
    {"type": "function", "function": {"name": "web_search", "description": "Search the web for up-to-date information.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "word_count", "description": "Count how many words are in a piece of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]
TOOLS = {"web_search": wc.web_search, "word_count": wc.word_count}


class WorkshopAgent(ResponsesAgent):
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        cfg = wc.get_llm_config()
        client = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)
        messages = [{"role": "system", "content": "Use your tools when they help answer accurately."}]
        messages += [{"role": m.role, "content": m.content} for m in request.input]
        for _ in range(5):
            msg = client.chat.completions.create(model=cfg.model, messages=messages, tools=SPECS).choices[0].message
            if not msg.tool_calls:
                break
            messages.append(msg)
            for call in msg.tool_calls:
                result = TOOLS[call.function.name](**json.loads(call.function.arguments or "{}"))
                messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
        item = self.create_text_output_item(text=msg.content or "", id=str(uuid.uuid4()))
        return ResponsesAgentResponse(output=[item])


mlflow.models.set_model(WorkshopAgent())

## 1. `01_trace_langgraph_agent.py`

MLflow 01 — see INSIDE your agent with tracing
MLflow is not an agent framework — it's the lab notebook for ANY framework:
  tracing (what happened, step by step), evaluation (is it good?), packaging/serving (ship it).
one line — mlflow.langchain.autolog() — records every LLM call, tool call and graph node.


other frameworks: mlflow.openai.autolog() (also covers the OpenAI Agents SDK), mlflow.crewai.autolog(),
                  mlflow.gemini.autolog(), mlflow.autogen.autolog(), mlflow.litellm.autolog() ...

In [ ]:
import mlflow
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

import workshop_common as wc

mlflow.set_tracking_uri("sqlite:///mlflow.db")      # a local file — no server needed
mlflow.set_experiment("SA-LIB Day2 agents")
mlflow.langchain.autolog()                          # <- that's it: everything below is traced

# ---- the same complete LangGraph agent as 01_langgraph/agent.py ----
cfg = wc.get_llm_config()
model = ChatOpenAI(model=cfg.model, base_url=cfg.base_url, api_key=cfg.api_key, temperature=0)


@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information."""
    return wc.web_search(query)


@tool
def word_count(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


tools = [web_search, word_count]
llm = model.bind_tools(tools)


def agent_node(state: MessagesState):
    return {"messages": [llm.invoke(state["messages"])]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")
agent = graph.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "chat-1"}}
for question in ["my name is m0h. search for the latest langchain version, then tell me how many words your answer is.",
                 "what's my name?"]:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
    print(result["messages"][-1].content)

# ---- check that the traces were recorded ----
mlflow.flush_trace_async_logging()
traces = mlflow.search_traces(max_results=2)
print(f"\nrecorded {len(traces)} trace(s). latest trace spans:")
for span in mlflow.get_trace(traces.iloc[0].trace_id).data.spans:
    print(f"  - {span.name} ({span.span_type})")
print("\nnow run:  mlflow ui --backend-store-uri sqlite:///mlflow.db   and open http://localhost:5000")

## 2. `02_trace_custom_agent.py`

MLflow 02 — tracing an agent you wrote yourself (no framework)
@mlflow.trace turns any function into a SPAN. nested calls become a tree.
mlflow.openai.autolog() adds a span for every LLM call (works with groq/deepseek/gemini too —
they all use the openai client).

In [ ]:
import json

import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

import workshop_common as wc

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("SA-LIB Day2 agents")
mlflow.openai.autolog()

cfg = wc.get_llm_config()
client = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)


@mlflow.trace(span_type=SpanType.TOOL)
def web_search(query: str) -> str:
    return wc.web_search(query)


@mlflow.trace(span_type=SpanType.TOOL)
def word_count(text: str) -> int:
    return wc.word_count(text)


TOOLS = {"web_search": web_search, "word_count": word_count}
SPECS = [
    {"type": "function", "function": {"name": "web_search", "description": "Search the web for up-to-date information.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "word_count", "description": "Count how many words are in a piece of text.",
     "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]


@mlflow.trace(span_type=SpanType.AGENT, name="my_agent")
def run_agent(question: str, max_steps: int = 5) -> str:
    messages = [{"role": "system", "content": "Use your tools when they help answer accurately."},
                {"role": "user", "content": question}]
    for _ in range(max_steps):                                     # plan -> act -> observe
        msg = client.chat.completions.create(model=cfg.model, messages=messages, tools=SPECS).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        for call in msg.tool_calls:
            result = TOOLS[call.function.name](**json.loads(call.function.arguments or "{}"))
            messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})
    return "stopped: too many steps"


print(run_agent("search for the latest langchain version, then tell me how many words your answer is."))

mlflow.flush_trace_async_logging()
trace = mlflow.get_trace(mlflow.get_last_active_trace_id())
print(f"\ntrace has {len(trace.data.spans)} spans:")
for span in trace.data.spans:
    print(f"  - {span.name} ({span.span_type})")

## 3. `03_evaluate_agent.py`

MLflow 03 — evaluating your agent
"it worked once on my laptop" is not a test. mlflow.genai.evaluate runs your agent on a small
dataset and scores every answer. here we use simple CODE scorers (free, no judge model needed).
(MLflow also has LLM-as-a-judge scorers like Correctness() and Safety() — see the README.)

In [ ]:
import mlflow
from mlflow.genai.scorers import scorer
from openai import OpenAI

import workshop_common as wc

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("SA-LIB Day2 agents")
mlflow.openai.autolog()

cfg = wc.get_llm_config()
client = OpenAI(base_url=cfg.base_url, api_key=cfg.api_key)


# the agent under test (kept tiny on purpose — swap in any agent from the other folders)
def answer(question: str) -> str:
    resp = client.chat.completions.create(model=cfg.model, temperature=0, messages=[
        {"role": "system", "content": "You are a concise assistant. Answer in at most 40 words."},
        {"role": "user", "content": question}])
    return resp.choices[0].message.content


# the test set: inputs + what we expect
dataset = [
    {"inputs": {"question": "What is the capital of Libya?"}, "expectations": {"must_contain": "Tripoli"}},
    {"inputs": {"question": "What does RAG stand for in AI?"}, "expectations": {"must_contain": "retrieval"}},
    {"inputs": {"question": "Explain what an AI agent is in two sentences."}, "expectations": {"must_contain": "tool"}},
]


# scorers: plain python functions that grade one answer
@scorer
def contains_expected(outputs: str, expectations: dict) -> bool:
    return expectations["must_contain"].lower() in outputs.lower()


@scorer
def is_concise(outputs: str) -> bool:
    return wc.word_count(outputs) <= 40


results = mlflow.genai.evaluate(data=dataset, predict_fn=answer, scorers=[contains_expected, is_concise])
print("\naggregate metrics:")
for name, value in sorted(results.metrics.items()):
    print(f"  {name}: {value}")

## 4. `04_package_agent.py`

MLflow 04 — package the agent as a versioned MLflow model, load it back, and serve it
logging = saving the code + dependencies + an example, with a version you can roll back to.

        (the exact command is printed at the end)
             -d '{"input": [{"role": "user", "content": "what is an AI agent?"}]}'

In [ ]:
from pathlib import Path

import mlflow

HERE = Path.cwd()
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("SA-LIB Day2 agents")

with mlflow.start_run(run_name="package-agent"):
    info = mlflow.pyfunc.log_model(
        name="workshop_agent",
        python_model=str(HERE / "agent_model.py"),            # "models from code": log the FILE
        code_paths=[str(HERE / "workshop_common.py")],         # helper module it imports
        input_example={"input": [{"role": "user", "content": "what is an AI agent?"}]},
        pip_requirements=["mlflow", "openai", "python-dotenv", "ddgs"],
    )
print("logged:", info.model_uri)

# load it back exactly like a deployed service would, and call it
agent = mlflow.pyfunc.load_model(info.model_uri)
response = agent.predict({"input": [{"role": "user", "content": "search for the latest langchain version"}]})
print(response["output"][0]["content"][0]["text"])

print(f'\nserve it:  mlflow models serve -m "{info.model_uri}" -p 5001 --env-manager local')

---
*SA-LIB AI Workshop 2026 · based on [build-ai-agents-free](https://github.com/Moh4696/build-ai-agents-free) by m0h (MIT)*